# Image Classification with MobileNetV3

This notebook classifies images using the **timm/mobilenetv3_sma11_100.1amb_inlk** model, a MobileNetV3 variant trained on ImageNet with 1000 classes.

In [2]:
import torch
from PIL import Image
import timm
import os
from pathlib import Path
import urllib.request

# Verifica se CUDA esta disponivel para aceleracao por GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

i:\github\gabrielsrs\nlw_operator\computer-vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [3]:
# Carrega o modelo pre-treinado do timm
model_name = "timm/mobilenetv3_small_100.lamb_in1k"
model = timm.create_model(model_name, pretrained=True)
model = model.eval()
model.to(device)

# Obtem a configuracao de transformacao de dados do modelo (normalizacao, redimensionamento, etc)
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

print(f"Model loaded: {model_name}")

# Download da lista de classes do ImageNet
class_index_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = urllib.request.urlopen(class_index_url)
class_names = [line.decode('utf-8').strip() for line in response.readlines()]
print(f"Loaded {len(class_names)} class names")

Model loaded: timm/mobilenetv3_small_100.lamb_in1k
Loaded 1000 class names


## Listing Images

We scan the `imagens/` directory for supported image files (.png, .jpg, .jpeg).

In [4]:
imagens_dir = Path("imagens/")
image_files = [f for f in os.listdir(imagens_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"Found {len(image_files)} images:")
for img in image_files:
    print(f"  - {img}")

Found 4 images:
  - bird.jpg
  - cats_dog.png
  - kitchen.png
  - pizza.png


## Classification

For each image:
1. Load and convert to RGB
2. Apply the model's preprocessing transforms
3. Run inference with no gradient computation
4. Convert logits to probabilities using softmax
5. Display top-5 predictions

In [5]:
for img_file in image_files:
    img_path = imagens_dir / img_file
    img = Image.open(img_path).convert('RGB')
    
    # Aplica as transformacoes e adiciona dimensao de batch
    input_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
    
    # Softmax para converter logits em probabilidades
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top5_prob, top5_idx = torch.topk(probabilities, 5)
    
    print(f"\n{'='*50}")
    print(f"Image: {img_file}")
    print(f"{'='*50}")
    for i in range(5):
        class_idx = top5_idx[i].item()
        class_name = class_names[class_idx]
        print(f"  {i+1}. {class_name}: {top5_prob[i].item():.4f}")


Image: bird.jpg
  1. bulbul: 0.1461
  2. chickadee: 0.1193
  3. junco: 0.0997
  4. indigo bunting: 0.0483
  5. jay: 0.0460

Image: cats_dog.png
  1. Rhodesian ridgeback: 0.1238
  2. basenji: 0.1206
  3. dingo: 0.0930
  4. redbone: 0.0686
  5. kelpie: 0.0358

Image: kitchen.png
  1. plate rack: 0.1120
  2. dining table: 0.1031
  3. altar: 0.0853
  4. microwave: 0.0508
  5. china cabinet: 0.0485

Image: pizza.png
  1. pizza: 0.9676
  2. French loaf: 0.0031
  3. pretzel: 0.0020
  4. hot pot: 0.0014
  5. soup bowl: 0.0011
